# Natural Language Processing (NLP) / Generative AI R&D
## Phone Exception Survey 
### Data Processing (Generative Query)

Read inputs from a series of PDFs and perform:

+ Data Save to a unified format
+ Sentiment Analysis and Sentiment Confidence
+ Categorization
+ All comments Word Cloud
+ All comments Summarization

Output presented to Power BI dashboard.

### Version History
+ v0.1 - Initial prototype, minimal to no data cleansing, inside source and reviewed by HITL.
+ v0.2 - Added code to read all PDF's, process generative results, use a data class, use PDF Miner to parse the form.  Pending join of data from PDF's into Panda's Data Frame.
+ v0.3 - Found multiple "forms" in the data payload, a resultant of "Unknown" in user name and user email means the "form" is not the standard "WO form" and is therefore invalid.  Created new DF output.
+ v0.4 - Fixed problem with CIO, SUpervisor and Boss fields not loading in datastructure.


### TODO
+ None

### Privacy Information

No action taken.

### Prompt Injection Defense

No action taken.

### References:
+  https://pdfminersix.readthedocs.io/en/latest/howto/acro_forms.html

### Technical References

### Jupyte Notebook Hints
+  https://jupyter-tutorial.readthedocs.io/en/24.1.0/notebook/shortcuts.html

In [1]:
# -*- coding: utf-8 -*-

### Environment Validation

Using GCP or Azure read in arrays representing minimal library requirements (which might not be present in a Google Colab environment) and install / load the libraries as required.  Additional imports for standard libraries and tailored content to follow.

In [2]:
#specific libraries like cudf and GPU enabled libs will not load via typical pip install efforts.
#use anaconda to install those capabilities into you env within the kernel you're using
#For CUDF within the desired Anaconda environment:
#    conda install -c rapidsai -c conda-forge -c nvidia rapids=24.06 

###########################################
#- Minimal imports to start
###########################################
try:
    import sys
    import subprocess
    import importlib.util
    import atexit
except ImportError as e:
    print("There was a problem importing the most basic libraries necessary for this code.")
    print(repr(e))
    raise SystemExit("Stop right there!")

###########################################
#- Final Exit Routine
###########################################
@atexit.register
def goodbye():
    print("GOODBYE")

###########################################
#- Cloud Environment Setup (Priming)
###########################################
# variables establishing environments
ENV_GCP=0
ENV_AZURE=1
user_input=-1
environments=["GCP", "Azure"]
    
#prompt user for environment before continuing
user_input = 0
while True:
  try:
     if user_input > -1:
         break;
     user_input = int(input("Select the environment you're running: (0) GCP (1) Azure"))     
     if user_input > 1:
         print("Not a valid choice, please try again.")
         continue;
  except ValueError:
     print("Not a valid choice, please try again.")
     continue
  else:
     print(f"Environment selected is: {environments[user_input]}")
     break 
        
############################################
#- Import a custom library, in this case a fairly useful logging framework
############################################
from pathlib import Path
debug_lib_location = Path("./")
sys.path.append(str(debug_lib_location))
try:
  import debug
  debug.msg_debug("...debug library loaded.")
except ImportError as e:
  print("There was a problem importing the debug library.")
  print(repr(e))
  raise SystemExit("Without the debug logging library, this code will not run.")

libraries=["transformers", "langchain", "openpyxl", "python-dotenv", "gensim", 
           "alive-progress", "tqdm", "pyspellchecker", "wordcloud", "langchain", "icecream", "numba", 
           "fitz","dataclasses", "commonregex", "transformers", "spacy", "PyMuPDF", "PyPDF2", "pdfminer", 
           "pdfplumber","pdf2image","pytesseract", "cupy" ]    
debug.msg_info(f"Validating environment for the following pip packages: {libraries}")

#load environment for non-generative libraries
try:
    for library in libraries:
      if library == "Pillow":
        spec = importlib.util.find_spec("PIL")
      else:
        spec = importlib.util.find_spec(library)
      if spec is None:
        print("...installing library " + library)
        subprocess.run(["pip", "install" , library, "--quiet"])
      else:
        print("...library " + library + " already installed.")
except (subprocess.CalledProcessError, Exception) as e:
    print("Error: Failed to install required packages, your code might not run properly.")
    print(repr(e))

#load environment specific libraries for generative AI.
try:    
    if environments[user_input]=="GCP":
      subprocess.run(["pip", "install" , "--upgrade", "google-cloud-aiplatform", "--quiet"])
      subprocess.run(["pip", "install" , "--upgrade", "google-cloud-secret-manager", "--quiet"])
      gcp_libraries=["google-generativeai","google.protobuf", "google.generativeai", "google.cloud.aiplatform_v1beta1",]
      for library in gcp_libraries:
        spec = importlib.util.find_spec(library)
        if spec is None:
          print("...installing library " + library)
          try:
              subprocess.run(["pip", "install" , library, "--quiet"])
          except (subprocess.CalledProcessError, Exception) as e:
              print("Error: Failed to install required packages, your code might not run properly.")
              print(repr(e))
        else:
          print("...library " + library + " already installed.")
    
        from google.cloud import aiplatform
        import vertexai.preview
        import vertexai
        import openai
        from google.auth import default, transport
        from google.cloud import secretmanager
        import google.generativeai as genai
        from vertexai.preview.generative_models import GenerativeModel
        from vertexai.preview.generative_models import GenerationConfig
        from google.cloud.aiplatform_v1beta1.types.openapi import Schema
        from google.cloud.aiplatform_v1beta1.types.openapi import Type
        from google.protobuf.json_format import MessageToDict        
        
          
    elif environments[user_input]=="Azure":
      azure_libraries=["openai", ]
      for library in azure_libraries:
        spec = importlib.util.find_spec(library)
        if spec is None:
          print("...installing library " + library)
          print("...installing library " + library)
          try:
              subprocess.run(["pip", "install" , library, "--quiet"])
          except (subprocess.CalledProcessError, Exception) as e:
              print("Error: Failed to install required packages, your code might not run properly.")
              print(repr(e))
        else:
          print("...library " + library + " already installed.")
    else:
        print("There was a problem processing your request.  Only numeric input of 0 or 1 is allowed.")
        print("Continued operations is not possible without the proper installed tools.")
        raise SystemExit("Stop right there!")
except Exception as e:
    print("There was a problem processing library installs for Generative AI libraries")
    print(repr(e))
    raise SystemExit("Stop right there!")

debug.msg_debug("...dynamic environment installs complete.")

[2024-11-01 16:25:16 UTC]   DEBUG: ...debug library loaded. 
[2024-11-01 16:25:16 UTC]    INFO: Validating environment for the following pip packages: ['transformers', 'langchain', 'openpyxl', 'python-dotenv', 'gensim', 'alive-progress', 'tqdm', 'pyspellchecker', 'wordcloud', 'langchain', 'icecream', 'numba', 'fitz', 'dataclasses', 'commonregex', 'transformers', 'spacy', 'PyMuPDF', 'PyPDF2', 'pdfminer', 'pdfplumber', 'pdf2image', 'pytesseract', 'cupy'] 
...library transformers already installed.
...library langchain already installed.
...library openpyxl already installed.
...installing library python-dotenv
...library gensim already installed.
...installing library alive-progress
...library tqdm already installed.
...installing library pyspellchecker
...library wordcloud already installed.
...library langchain already installed.
...library icecream already installed.
...library numba already installed.
...library fitz already installed.
...library dataclasses already installed.
...lib

### Additional Libraries

Read in core libraries, local logging framework, data science tools, Natural Language Processing Toolkit (NLTK), Spacy and other tools to support the effort.

In [3]:
debug.msg_info("Library imports")    
############################################
# INCLUDES
############################################

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# a set of libraries that perhaps should always be in Python source
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...core libraries.")
import os
import datetime
from dateutil import parser
import gc
import socket
import sys
import getopt
import inspect
import traceback
import warnings
import json
import pickle
from pathlib import Path
import itertools
import datetime
import re
import shutil
import string
from io import StringIO
import tqdm


import io
import math
import textwrap
import random
import glob
import time
from time import perf_counter
import subprocess
from multiprocessing import Pool
import backoff                    #annotation to support repeat calls on api failure


from unidecode import unidecode   #to handle strange characters
from dotenv import load_dotenv    #load environment vars/keys/etc...

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Function Profiling
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import cProfile
import pstats
import io
from pstats import SortKey

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# MS Excel Libraries
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
from openpyxl import Workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, GradientFill
from openpyxl.styles import Border, Side
from openpyxl.styles import Alignment
from openpyxl.styles import Font

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Data Science Libraries
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...classic data science libraries.")

#optimization routines
from numba import jit
import numpy as np
import scipy as sp
#from sklearn.linear_model import LinearRegression


# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Additional libraries for this work
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...application specific libraries.")
import math
from base64 import b64decode
from IPython.display import Image
import requests
from bs4 import BeautifulSoup                 #used to parse the text
from wordcloud import WordCloud, STOPWORDS    #custom library specifically designed to make word clouds
from spellchecker import SpellChecker

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- PDF Libraries
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import fitz
from pdfminer.pdfparser import PDFParser
from pdfminer.pdfdocument import PDFDocument
from pdfminer.pdftypes import resolve1
from pdfminer.psparser import PSLiteral, PSKeyword
from pdfminer.utils import decode_text

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Graphics
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...graphics.")
#import PIL
from PIL import Image
import PIL.ImageOps
import matplotlib as matplt
import matplotlib.pyplot as plt

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# progress bar
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...progress bars.")
from alive_progress import alive_bar
#from alive_progress.styles import showtime, Show
from tqdm.notebook import trange, tqdm
#from tqdm import trange, tqdm

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- PII libraries (regular expressions)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...regular expressions for PII and transformers for prompt injection defense.")
from commonregex import CommonRegex
from commonregex import email
from commonregex import time
from commonregex import credit_card
from commonregex import ip
from commonregex import ipv6
from commonregex import link
from commonregex import phone
from commonregex import street_address
from commonregex import btc_address

debug.msg_debug("...spacy (pii defense).")
import spacy
from spacy.language import Language
from spacy.tokens import Doc

debug.msg_debug("...hugging face model support.")
#injection defense
from transformers import pipeline

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- Tensorflow AI/ML libraries (seek to use GPU's)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#load first
try:
    debug.msg_debug("...TensorRT")    
    import tensorrt
    assert tensorrt.Builder(tensorrt.Logger())
except ImportError as ie:
    debug.msg_warning("Failed to import tensorrt, this might be a problem if trying for enhanced processing.")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    #load second
    debug.msg_debug("...TensorFlow")        
    import tensorflow as tf
except ImportError as ie:
    debug.msg_warning("Failed to import tensorflow, might not have a GPU or the proper environment loaded")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    debug.msg_debug("...CUDF")    
    import cudf
except ImportError as ie:
    debug.msg_warning("Failed to import cudf, likely don't have a GPU")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    debug.msg_debug("...Torch")    
    import torch
except ImportError as ie:
    debug.msg_warning("Failed to import torch, likely don't have a GPU or access to that library.")
    debug.msg_warning(f"...{repr(ie)}")
    pass

debug.msg_debug("...Pandas")    
import pandas as pd

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- NLTK required resources
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...natural language processing.")
import nltk
from nltk.stem import PorterStemmer  # A word stemmer based on the Porter stemming algorithm.  Porter, M. "An algorithm for suffix stripping." Program 14.3 (1980): 130-137.
from nltk import pos_tag
from nltk.tree import tree
#from nltk.book import *
from nltk import FreqDist
from nltk import sent_tokenize, word_tokenize
from nltk.corpus import stopwords    
from nltk.stem.wordnet import WordNetLemmatizer


nltk.download('punkt')
nltk.download("words")
nltk.download("stopwords")
nltk.download('wordnet')  
nltk.download('omw-1.4')  
#nltk.download('averaged_perceptron_tagger')      #looks like you have to download select neural layers for specific functions, head to read the erorr output to learn this.

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- Topic Modeling
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
from gensim import corpora
from gensim.models import LsiModel
from gensim.models import LdaModel



[2024-11-01 16:25:31 UTC]    INFO: Library imports 
[2024-11-01 16:25:31 UTC]   DEBUG: ...core libraries. 
[2024-11-01 16:25:31 UTC]   DEBUG: ...classic data science libraries. 
[2024-11-01 16:25:31 UTC]   DEBUG: ...application specific libraries. 
[2024-11-01 16:25:31 UTC]   DEBUG: ...graphics. 
[2024-11-01 16:25:31 UTC]   DEBUG: ...progress bars. 
[2024-11-01 16:25:31 UTC]   DEBUG: ...regular expressions for PII and transformers for prompt injection defense. 
[2024-11-01 16:25:31 UTC]   DEBUG: ...spacy (pii defense). 
[2024-11-01 16:25:34 UTC]   DEBUG: ...hugging face model support. 


/opt/conda/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2024-11-01 16:25:34.735469: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-01 16:25:34.756962: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-01 16:25:34.763541: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-01 16:25:34.780345: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimiz

[2024-11-01 16:25:36 UTC]   DEBUG: ...TensorRT 


/opt/conda/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


[2024-11-01 16:25:39 UTC]   DEBUG: ...TensorFlow 
[2024-11-01 16:25:39 UTC]   DEBUG: ...CUDF 
[2024-11-01 16:25:39 UTC] WARNING: Failed to import cudf, likely don't have a GPU 
[2024-11-01 16:25:39 UTC] WARNING: ...ModuleNotFoundError("No module named 'cudf'") 
[2024-11-01 16:25:39 UTC]   DEBUG: ...Torch 
[2024-11-01 16:25:39 UTC]   DEBUG: ...Pandas 
[2024-11-01 16:25:39 UTC]   DEBUG: ...natural language processing. 


[nltk_data] Downloading package punkt to /home/jupyter/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package words to /home/jupyter/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/jupyter/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jupyter/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jupyter/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Functions

In [4]:
def set_library_configuration() -> None:
    
    ############################################
    #- JUPYTER NOTEBOOK OUTPUT CONTROL / FORMATTING
    ############################################
    #pandas set floating point to 4 places to things don't run loose
    debug.msg_info("Setting Pandas and Numpy library options.")    
    #pd.set_option('display.max_colwidth', 10) # None if you want to view the full json blob in the printed dataframe, use this
    pd.set_option('display.max_colwidth', None) # None if you want to view the full json blob in the printed dataframe, use this
    pd.options.display.float_format = '{:,.4f}'.format
    np.set_printoptions(precision=4)

In [5]:
def profile_function(func):
    def wrapper(*args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()
        result = func(*args, **kwargs)
        pr.disable()
        s = io.StringIO()
        sortby = SortKey.CUMULATIVE
        ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
        ps.print_stats()
        print(s.getvalue())
        return result
    return wrapper

### Configure Tailored Output Configuration

In [6]:
############################################
#- JUPYTER NOTEBOOK OUTPUT CONTROL / FORMATTING
############################################
#pandas set floating point to 4 places to things don't run loose
debug.msg_info("Setting Pandas and Numpy library options.")    
pd.set_option('display.max_colwidth', 10) # None if you want to view the full json blob in the printed dataframe, use this
pd.options.display.float_format = '{:,.4f}'.format
np.set_printoptions(precision=4)

[2024-11-01 16:25:39 UTC]    INFO: Setting Pandas and Numpy library options. 


#### Privacy Information Defense

Look for Spacy entities known as "PERSON" for names of people and using regular expression library to find related PII information and abstract.

#### String Manipulation (Data String Cleanup Routines)

In [7]:
## Read the string and clean up the markdown created by gemini
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @param (Text for type to transform) - str- String type
#  Hat tip to Yvan@google
def markdown_escaper(text: str, type: str = 'json'):
    escape_length = len(f'```{type}')
    if text[:escape_length] == f'```{type}' and text[-3:] == '```':
        return text[escape_length:-3]
    else:
        return text

## Read the markdown provided and turn it into JSON
#
#  @param (Text for text to clean) - str    - Markdown to transform
#  Hat tip to Yvan@google
def markdown_to_json(md: str):
    return json.loads(markdown_escaper(md))

## Read the contents of a text input and anything that doesn't support a potential date format
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
def clean_date(inc_str: str) -> str:
    resumeText = re.sub('httpS+s*', ' ', inc_str)  # remove URLs
    resumeText = re.sub('RT|cc', ' ', resumeText)  # remove RT and cc
    resumeText = re.sub('#S+', '', resumeText)  # remove hashtags
    resumeText = re.sub('@S+', '  ', resumeText)  # remove mentions
    resumeText = re.sub(r'\r', '', resumeText)
    resumeText = re.sub(r'\n', '', resumeText)
    resumeText = re.sub(' +', ' ', resumeText) # remove extra whitespace
    resumeText = re.sub(r'\t', ' ', resumeText) #remove tabs

    resumeText.rstrip()
    resumeText.lstrip()
    resumeText = re.sub('[%s]' % re.escape("""!"#$%&'()*+,.:;<=>?@[]^_`{|}~"""), ' ', resumeText)  # remove punctuations

    resumeText=re.sub(r'\W+', ' ', resumeText)
    resumeText=re.sub(' +', ' ', resumeText)       
    return resumeText


## Read the contents of a text input and remove URL's, extra spaces, carriage returns, etc..
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
def clean_text(inc_str: str) -> str:
    resumeText = re.sub('httpS+s*', ' ', inc_str)  # remove URLs
    resumeText = re.sub('RT|cc', ' ', resumeText)  # remove RT and cc
    resumeText = re.sub('#S+', '', resumeText)  # remove hashtags
    resumeText = re.sub('@S+', '  ', resumeText)  # remove mentions
    resumeText = re.sub(r'\r', '', resumeText)
    resumeText = re.sub(r'\n', '', resumeText)
    resumeText = re.sub(' +', ' ', resumeText) # remove extra whitespace
    resumeText = re.sub(r'\t', ' ', resumeText) #remove tabs

    resumeText.rstrip()
    resumeText.lstrip()
    resumeText = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[]^_`{|}~"""), ' ', resumeText)  # remove punctuations
    resumtText = ''.join([i if ord(i) < 128 else ' ' for i in resumeText])

    resumeText=re.sub(r'\W+', ' ', resumeText)
    resumeText=re.sub(' +', ' ', resumeText)       
    # Remove punctuation
    #no_punctuation = (nopunc.translate(str.maketrans('', '', string.punctuation)) for nopunc in lower)
    #resumeText = ''.join(x for x in resumeText if x.isalnum())
    #resumeText = re.sub(r'[^x00-x7f]',r' ', resumeText) 
    #resumeText = re.sub('s+', ' ', resumeText)  # remove extra whitespace
    return resumeText

## Read the contents of a text input and remove URL's, extra spaces, carriage returns, etc..
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - All junk removed.
def clean_json(inc_str: str) -> str:
    resumeText = re.sub(r'\r', '', inc_str)
    resumeText = re.sub(r'\n', '', resumeText)
    resumeText = re.sub(' +', ' ', resumeText) # remove extra whitespace
    resumeText = re.sub(r'\t', ' ', resumeText) #remove tabs
    resumeText.rstrip()
    resumeText.lstrip()
    resumeText = re.sub('[%s]' % re.escape("""#*'"""), ' ', resumeText)  # remove punctuations
    return resumeText

## Read the contents of a text input remove stop words (nltk)
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Stop words removed
def clean_stop_words(inc_str:str) -> str:
        filtered_list = []
        response=word_tokenize(inc_str)
        wordlist = [x for x in response if (len(x)>=2 and x.isalpha())]
        for word in wordlist:
            if word.casefold() not in stop_words:
              filtered_list.append(word)
        return str(' '.join(filtered_list))

## Read the contents of a text input and remove extra spaces
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Extra spaces removed
def clean_string (inc_str:str) -> str:
        #response=re.sub(r'\W+', ' ', inc_str)
        response=re.sub(' +', ' ', inc_str) 
        response = re.sub(r'\r', '', response)
        response = re.sub(r'\n', '', response)
        response = re.sub(r'\t', ' ', response) #remove tabs
        response = response.rstrip()
        response = response.lstrip()
        
        return str(response)

## Read the contents of a text input modify string for stem words (nltk)
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Stem words altered
def clean_stem_words(inc_str:str) -> str:
        filtered_list = []
        response=word_tokenize(inc_str)
        wordlist = [x for x in response if (len(x)>=2 and x.isalpha())]
        stemmed_words = [stemmer.stem(word) for word in wordlist]
        return str(' '.join(stemmed_words))

## Read the contents of a text input remove stop words (nltk)
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Stop words removed
def clean_lemmatizer_words(inc_str:str) -> str:
        filtered_list = []
        response=word_tokenize(inc_str)
        wordlist = [x for x in response if (len(x)>=2 and x.isalpha())]
        lemmatized_words = [lemmatizer.lemmatize(word) for word in wordlist]
        return str(' '.join(lemmatized_words))

## Read the contents of a text cleanse the string
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Complete purge of all content for NLP
def cleanse_string(inc_str: str) -> str:
    response=clean_string(inc_str)
    response=clean_text(response)
    response=clean_stop_words(response)
    response=clean_stem_words(response)
    response=clean_lemmatizer_words(response)
    response=word_tokenize(response)
    return response

In [8]:
## Decodes values that are PDFMiner explicit types into something more manageable
# @param obj
#
def decode_value(value):

    response=""
    # decode PSLiteral, PSKeyword
    if isinstance(value, (PSLiteral, PSKeyword)):
        response = value.name

    # decode bytes
    if isinstance(value, bytes):
        response = decode_text(value)

    return response


## Function Declaration

#### Custom Exception Display

In [9]:
## Manages exception output.
#  @param   (Exception)             - Exception to expound upon
def process_exception(inc_exception) -> None:
    print(f"{BOLD_START}(Exception encountered):{BOLD_END} {type(inc_exception).__name__}")
    print(f"Details: {str(inc_exception)}")
    print("Traceback:")
    traceback.print_exc()

#### Library Manifest Display

In [10]:
## Outputs library version history of effort.
#
#  @returns (None)                  - None
def lib_diagnostics() -> None:

    import pkg_resources
    
    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}") 
    
    package_name_length=40
    package_version_length=20

    # Get installed packages
    the_packages=["cupy", "jupyter-core", "langchain", "langchain-core", "nltk", "numba", "numpy", "pandas", "pydantic", "pyspellchecker", "spacy", "scipy", "scikit-learn", "seaborn", "usaddress", "xarray",]
    the_packages.sort()
    
    installed_dict = {pkg.key: pkg.version for pkg in pkg_resources.working_set}
    installed=list(installed_dict.keys())
    installed.sort()
    
    #for package_idx, package_name in enumerate(installed):
    for idx, name in enumerate(installed):
         if name in the_packages:
             installed_version = installed_dict[name]
             print(f"{name:<40}#: {str(pkg_resources.parse_version(installed_version)):<20}")
   
    try:
        print(f"{'TensorFlow version':<40}#: {str(tf.__version__):<20}")
        print(f"{'     gpu.count:':<40}#: {str(len(tf.config.experimental.list_physical_devices('GPU')))}")
        print(f"{'     cpu.count:':<40}#: {str(len(tf.config.experimental.list_physical_devices('CPU')))}")
    except Exception as e:
        pass

    try:
        print(f"{'Torch version':<40}#: {str(torch.__version__):<20}")
        print(f"{'     GPUs available?':<40}#: {torch.cuda.is_available()}")
        print(f"{'     count':<40}#: {torch.cuda.device_count()}")
        print(f"{'     current':<40}#: {torch.cuda.current_device()}")
    except Exception as e:
        pass


    try:
      print(f"{'OpenAI Azure Version':<40}#: {str(the_openai_version):<20}")
    except Exception as e:
      pass

    print(f"{BOLD_START}List Devices{BOLD_END} #########################################")
    try:
      from tensorflow.python.client import device_lib
      print(device_lib.list_local_devices())
      print("")
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(åe)))

    print(f"{BOLD_START}Devices Counts{BOLD_END} ########################################")
    try:
      print(f"Num GPUs Available: {str(len(tf.config.experimental.list_physical_devices('GPU')))}" )
      print(f"Num CPUs Available: {str(len(tf.config.experimental.list_physical_devices('CPU')))}" )
      print("")
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(e)))

    print(f"{BOLD_START}Optional Enablement{BOLD_END} ####################################")
    try:
      gpus = tf.config.experimental.list_physical_devices('GPU')
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(e)))

    if gpus:
      # Restrict TensorFlow to only use the first GPU
      try:
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print( str( str(len(gpus)) + " Physical GPUs," + str(len(logical_gpus)) + " Logical GPU") )
      except RuntimeError as e:
        # Visible devices must be set before GPUs have been initialized
        print(str(repr(e)))
      print("")
        
    debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}") 
    return

In [11]:
def output_csv(inc_filename:str, inc_df: pd.DataFrame) -> None:
    
    output_filename=inc_filename
    debug.msg_debug(f"Saving the data to a file ({output_filename}).")
    inc_df.to_csv(output_filename, sep=DELIM, header=True, index=False)
    

In [12]:
def output_excel(inc_filename:str, inc_df: pd.DataFrame) -> None:
    
    output_filename=inc_filename
    debug.msg_debug(f"Saving the data to a file ({output_filename}).")
    with pd.ExcelWriter(inc_filename, mode='w') as writer:  
        inc_df.to_excel(writer)
    

In [13]:
def output_data(data_version_release: str, inc_df:pd.DataFrame)-> None:
    #save to textual output
    target_filename=f"./{data_version_release}"+"_output.bin"    
    try:    
        pickle.dump(inc_df, open(target_filename, "wb"))
        me = pickle.load(open(target_filename, "rb"))
    except (pickle.UnpicklingError, FileNotFoundError, IOError, Exception)  as e:
        debug.msg_warning("FAILED to unpickle the saved binary file, you might have corruption, investigate.")
        process_exception(e)
    debug.msg_debug(f"...saved and reloaded {target_filename}")
            
    target_filename=f"./{data_version_release}"+"_output.csv"    
    try:
        output_csv(target_filename, inc_df)
    except (pickle.UnpicklingError, FileNotFoundError, IOError, Exception)  as e:    
        debug.msg_warning("FAILED to process the file, you might have corruption, investigate.")
        debug.msg_warning(f"...target output filename: {target_filename}")
        process_exception(e)

    target_filename=f"./{data_version_release}"+"_output.xlsx"    
    #save to MS Excel
    try:
        output_excel(target_filename, inc_df)
    except (pickle.UnpicklingError, FileNotFoundError, IOError, Exception)  as e:    
        debug.msg_warning("FAILED to process the file, you might have corruption, investigate.")
        debug.msg_warning(f"...target output filename: {target_filename}")            
        process_exception(e)
        

In [14]:
## Read the contents of a resume in PDF format. Unstructured data.
#  https://towardsdatascience.com/extracting-text-from-pdf-files-with-python-a-comprehensive-guide-9fc4003d517
#
#  @param (Text for filename, str) - str    - Filename for target resume.
def read_pdf(inc_filename:str) -> dict:

    data={}
    data.update({"id" : os.path.basename(inc_filename)})
    
    try:
        if not ( os.path.isfile(inc_filename) ):
            debug.msg_error(f"ERROR detected, the input data file for work further in the notebook is missing.  Aborting execution.")
            debug.msg_error(f"  Resolve the {inc_filename} missing file and repeat.")
            #raise SystemExit("Unable to continue without data.")
            return data
    except Exception as e:
        process_exception(f"ERROR detected trying detect the PDF path as follows: {str(e)}.  Returning user object with just the file Id registered.")
        return data

    try:
        with open(inc_filename, "rb") as my_file:        
            pdf_parser = PDFParser(my_file)
            doc = PDFDocument(pdf_parser)
            res = resolve1(doc.catalog)
            
            if 'AcroForm' not in res:
                debug.msg_warning("No AcroForm Found, unable to process this PDF.  Returning the user object with just the file Id registered.")
                return data

            fields = resolve1(doc.catalog['AcroForm'])['Fields']  # may need further resolving

            for f in fields:
                field = resolve1(f)
                name, values = field.get('T'), field.get('V')
                #print(name,values)

                # decode name
                if name is not None:
                    the_name = clean_string(decode_text(name)).lower()
                else:
                    continue
                    
                # resolve indirect obj
                value=""
                values = resolve1(values)                
                if values != None:
                    # decode value(s)
                    if isinstance(values, list):
                        values = [clean_string(decode_value(v)).lower() for v in values]
                    elif isinstance(values, dict):
                        #we're not processing dictionaries which appear to be the signature block for PKI stuff.
                        continue
                    else:
                        values = clean_string(decode_value(values)).lower()
                    value=" ".join([values]).lower()
                else:
                    value="".lower()
                
                data.update({the_name: value})

    except (IOError, Exception) as e:
        process_exception(f"ERROR to read the datafile provided, likely a malformed PDF file: {str(e)}")
        return data

    return data


In [15]:
## Main routine that reads each PDF (parallel or individual), pulls out data for read_pdf into a user_exemption data class and stores it.
#  Variable self-contained.
#  @param (None)

def process_pdfs() -> list:
    
    debug.msg_info(f"Entered {__name__} {inspect.stack()[0][3]}")
    phone_exceptions_file_list=[]
    phone_exceptions=[]
    
    #CGW, TODO, FIX
    target_folder=DATA_DIR+os.sep+"phone_exceptions"+os.sep+"ExceptionForms"+os.sep
    #target_folder="./"


    if os.path.exists(target_folder):
        for file in os.listdir(target_folder):
            if file.endswith(".pdf") or file.endswith(".PDF"):
                phone_exceptions_file_list.append(os.path.join(target_folder, file))
    else:
        debug.msg_error(f"Directory {target_folder} not found, likely won't get any data.")

    debug.msg_debug(f"{len(phone_exceptions_file_list)} files read in.")
    
    
    debug.msg_debug("Reading exception forms...")
    start_t=perf_counter()    

    with Pool() as pool:
        results = pool.map(read_pdf, phone_exceptions_file_list)
        for idx, user_exception in enumerate(results):
            phone_exceptions.append(user_exception)
    """
    for idx, filename in enumerate(phone_exceptions_file_list):
        results=read_pdf(filename)
        phone_exceptions.append(results)
        if idx > 100:
           break
    """    
    
    end_t=perf_counter()         
    print(f"Total processing time is:{end_t-start_t}")

    
    #for the_exemption in phone_exceptions:
    #    print(f"Id: {the_exemption.id} - Multi-Device? {str(the_exemption.multidev)} - Non-Standard: {str(the_exemption.nonstddev)} - Name:{the_exemption.name}")

    debug.msg_info(f"Exited {__name__} {inspect.stack()[0][3]}")
       
    return phone_exceptions



In [16]:
## Main routine that executes all code, does return a data frame of data for further analysis if desired.
#
#  @param (list)
def process_exemptions(inc_exemptions: list) -> pd.DataFrame:

    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")

    #establish data version, aligned with code
    data_version_release="-".join([str(VERSION_NAME), str(VERSION_MAJOR), str(VERSION_MINOR), str(VERSION_RELEASE)])

    field_name_set=set()
    dataset={}
 
    for idx,exemption in enumerate(inc_exemptions):
        field_name_set.update(exemption.keys())
        #print("-------------------------------------------------------------------------------")                    
        #for the_key in exemption.keys():
        #    print(f"{the_key}:{exemption[the_key]}")
        #print("-------------------------------------------------------------------------------")
 

    #print("")
    #print("")
    #print("FIELD NAME SET - ########################################################")
    #print(field_name_set)        
    #print("#########################################################################")
    #print("")
    #print("")
        
    for idx,exemption in enumerate(inc_exemptions):
        the_id=exemption["id"]
        the_record={}
        for field_name in field_name_set:
            if field_name in exemption:
                the_record.update({field_name : exemption[field_name] })
            else:
                the_record.update({field_name : "NO MATCH"})
        dataset[the_id]=the_record

    #print("DATASET - ###############################################################")           
    #print(dataset)
    #print("#########################################################################")
    #print("")
    #print("")
    
    df = pd.DataFrame.from_dict(dataset)
    df = df.transpose()
    #df = pd.DataFrame()
    debug.msg_info(f"Exited {__name__} {inspect.stack()[0][3]}")
 
    return df




### Pythonic way of calling main if we go to a script

#### Main Routine (call all other routines)

In [17]:
if __name__ == "__main__":

    set_library_configuration()
    start_t=perf_counter()
    print("BEGIN PROGRAM")

    ############################################
    # SECRETS & ENV VARIABLES
    ############################################
    load_dotenv()

    ############################################
    # GLOBAL CONFIGURATION
    ############################################
    #used for values outside standard ASCII, just do it, you'll need it
    ENCODING  ="utf-8"
    os.environ['PYTHONIOENCODING']=ENCODING
    #spacy requirement
    os.environ['TOKENIZERS_PARALLELISM']="false"

    debug.msg_info("Variable declaration.")    
    ############################################
    # GLOBAL VARIABLES
    ############################################
    DEBUG = 1
    DEBUG_DATA = 0
    
    # CODE CONSTRAINTS
    VERSION_NAME    = "PHNEXP_UNRAVEL"
    VERSION_MAJOR   = 0
    VERSION_MINOR   = 1
    VERSION_RELEASE = 0
    
    #used for values outside standard ASCII, just do it, you'll need it
    ENCODING  ="utf-8"
    TEXT_WIDTH=77
    BOLD_START = "\033[1m"
    BOLD_END = "\033[0;0m"
    
    ###########################################
    #- API Parameters for things like WordCloud
    ###########################################
    IMG_BACKGROUND=None                        #None without quotes or "black", "white", etc...
    IMG_FONT_SIZE_MIN=14
    IMG_WIDTH=800
    IMG_HEIGHT=600
    
    ############################################
    # APPLICATION VARIABLES
    ############################################
    PROJECT_ID= "usfs-gcp-rand-test-3"
    BUCKET_ID = "usfs-gcp-rand-test3-data-usc1"
    LOCATION = "us-central1"
    SPELL_CHECK_DISTANCE=2
    MINIMUM_AI_WAIT=1                         #seconds
    os.environ["MINIMUM_AI_WAIT"] = "str(MINIMUM_AI_WAIT)"    
    DATA_DIR = f"/home/jupyter/projects/data/{BUCKET_ID}/source_data/nlp"
    EXCEL_CHAR_BOUNDARY=90
    DELIM="^"
    
      
    #setup the text wrapper
    debug.msg_debug(f"...Text Wrapper instantiated.")
    wrapper = textwrap.TextWrapper(width=TEXT_WIDTH)
    
    #show your libraries
    lib_diagnostics()


    ###########################################
    #- Workhorse routine
    ###########################################
    #process_pdfs()
    df=process_exemptions(process_pdfs())

    ##########################################################
    # Save results - final save
    ##########################################################        
    #establish data version, aligned with code
    data_version_release="-".join([str(VERSION_NAME), str(VERSION_MAJOR), str(VERSION_MINOR), str(VERSION_RELEASE)])
    save_filename=f"{data_version_release}"
    output_data(save_filename, df)

    
    end_t=perf_counter()
    print("END PROGRAM")
    print(f"Elapsed time: {end_t - start_t}")


[2024-11-01 16:25:39 UTC]    INFO: Setting Pandas and Numpy library options. 
BEGIN PROGRAM
[2024-11-01 16:25:39 UTC]    INFO: Variable declaration. 
[2024-11-01 16:25:39 UTC]   DEBUG: ...Text Wrapper instantiated. 


/var/tmp/ipykernel_2739175/3795858203.py:6: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


[2024-11-01 16:25:40 UTC]    INFO: Entering __main__ lib_diagnostics 
jupyter-core                            #: 5.7.2               
langchain                               #: 0.3.1               
langchain-core                          #: 0.3.6               
nltk                                    #: 3.9.1               
numba                                   #: 0.60.0              
numpy                                   #: 1.26.4              
pandas                                  #: 2.2.3               
pydantic                                #: 2.9.2               
pyspellchecker                          #: 0.8.1               
scikit-learn                            #: 1.5.2               
scipy                                   #: 1.13.1              
seaborn                                 #: 0.13.2              
spacy                                   #: 3.7.6               
usaddress                               #: 0.5.10              
TensorFlow version                

I0000 00:00:1730478340.330009 2739175 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730478340.333954 2739175 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730478340.336017 2739175 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730478340.364130 2739175 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

[2024-11-01 16:25:46 UTC] WARNING: No AcroForm Found, unable to process this PDF.  Returning the user object with just the file Id registered. 
[2024-11-01 16:25:47 UTC] WARNING: No AcroForm Found, unable to process this PDF.  Returning the user object with just the file Id registered. 
[2024-11-01 16:25:52 UTC] WARNING: No AcroForm Found, unable to process this PDF.  Returning the user object with just the file Id registered. 
[2024-11-01 16:25:55 UTC] WARNING: No AcroForm Found, unable to process this PDF.  Returning the user object with just the file Id registered. 
[2024-11-01 16:25:56 UTC] WARNING: No AcroForm Found, unable to process this PDF.  Returning the user object with just the file Id registered. 
Total processing time is:18.88763071698486
[2024-11-01 16:25:59 UTC]    INFO: Exited __main__ process_pdfs 
[2024-11-01 16:25:59 UTC]    INFO: Entering __main__ process_exemptions 
[2024-11-01 16:25:59 UTC]    INFO: Exited __main__ process_exemptions 
[2024-11-01 16:25:59 UTC]   